In [ ]:
import os
import re
import time
import json
import random
import hashlib
import subprocess
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
PIKAFISH_PATH = r"pikafish.exe"   # change this
INPUT_CSV = r"datasetv2_train.csv"  # or val csv
SOURCE_GAMES_TXT = r"xqdb_masters_40711_UCI_games.txt"

OUTPUT_DIR = Path("sampled_xiangqi_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_RAW_CSV = OUTPUT_DIR / "datasetv2_labeled_raw.csv"
OUTPUT_NORM_CSV = OUTPUT_DIR / "datasetv2_labeled_normalized.csv"

OUTPUT_RAW_NPZ = OUTPUT_DIR / "datasetv2_labeled_raw.npz"
OUTPUT_NORM_NPZ = OUTPUT_DIR / "datasetv2_labeled_normalized.npz"

# engine settings
ENGINE_THREADS = 6
ENGINE_HASH_MB = 256
LABEL_DEPTH = 10
LABEL_MAX_NODES = 50000

# label processing
RAW_CLIP_CP = 3000
NORMALIZATION_MODE = "tanh"   # "none", "clip", "smooth", "tanh"
TANH_SCALE = 800.0
SMOOTH_SCALE = 1000.0
TARGET_RANGE = 15.0

# plotting
NUM_HIST_BINS = 60

In [ ]:
def parse_pgn_style_uci_text(text: str):
    lines = text.splitlines()

    games = []
    current_headers = {}
    current_move_lines = []

    def flush_game():
        nonlocal current_headers, current_move_lines, games

        if not current_headers and not current_move_lines:
            return

        move_text = " ".join(line.strip() for line in current_move_lines if line.strip())
        moves = re.findall(r'\b[a-i][0-9][a-i][0-9]\b', move_text.lower())

        if len(moves) > 0:
            games.append({
                "headers": current_headers.copy(),
                "moves": moves
            })

        current_headers = {}
        current_move_lines = []

    for raw_line in lines:
        line = raw_line.strip()

        if not line:
            continue

        if line.startswith("[") and line.endswith("]"):
            if len(current_move_lines) > 0:
                flush_game()

            m = re.match(r'^\[(.+?)\s+"(.*)"\]$', line)
            if m:
                key = m.group(1).strip()
                value = m.group(2).strip()
                current_headers[key] = value
            continue

        current_move_lines.append(line)

    flush_game()
    return games

with open(SOURCE_GAMES_TXT, "r", encoding="utf-8") as f:
    source_text = f.read()

source_games = parse_pgn_style_uci_text(source_text)

print("total source games:", len(source_games))
if len(source_games) > 0:
    print("example headers:", source_games[0]["headers"])
    print("example moves:", source_games[0]["moves"][:10])

In [ ]:
df = pd.read_csv(INPUT_CSV)

print("csv rows:", len(df))
print("unique game_index:", df["game_index"].nunique())
display(df.head())

In [ ]:
FILES = "abcdefghi"
FILE_TO_X = {c: i for i, c in enumerate(FILES)}

PIECE_TO_CHANNEL = {
    "rk": 0, "rr": 1, "rh": 2, "re": 3, "ra": 4, "rc": 5, "rp": 6,
    "bk": 7, "br": 8, "bh": 9, "be": 10, "ba": 11, "bc": 12, "bp": 13
}


def initial_board():
    b = [["." for _ in range(9)] for _ in range(10)]

    # Red (bottom)
    b[0] = ["rr", "rh", "re", "ra", "rk", "ra", "re", "rh", "rr"]
    b[2][1], b[2][7] = "rc", "rc"
    for x in [0, 2, 4, 6, 8]:
        b[3][x] = "rp"

    # Black (top)
    b[9] = ["br", "bh", "be", "ba", "bk", "ba", "be", "bh", "br"]
    b[7][1], b[7][7] = "bc", "bc"
    for x in [0, 2, 4, 6, 8]:
        b[6][x] = "bp"

    return b


def clone_board(board):
    return [row[:] for row in board]


def apply_move(board, move: str):
    sx, sy = FILE_TO_X[move[0]], int(move[1])
    dx, dy = FILE_TO_X[move[2]], int(move[3])

    board[dy][dx] = board[sy][sx]
    board[sy][sx] = "."


def board_to_tensor(board, side_to_move: str):
    x = np.zeros((15, 10, 9), dtype=np.float32)

    for y in range(10):
        for col in range(9):
            p = board[y][col]
            if p != ".":
                x[PIECE_TO_CHANNEL[p], y, col] = 1.0

    x[14, :, :] = 1.0 if side_to_move == "red" else 0.0
    return x


def side_to_move_from_ply_index(ply_index_0based: int) -> str:
    return "red" if (ply_index_0based % 2 == 0) else "black"


def opposite_side(side: str) -> str:
    return "black" if side == "red" else "red"


def reconstruct_position_before_ply(game_moves, ply_index_0based: int):
    board = initial_board()
    moves_prefix = []

    for i in range(ply_index_0based):
        mv = game_moves[i]
        apply_move(board, mv)
        moves_prefix.append(mv)

    side_before = side_to_move_from_ply_index(ply_index_0based)
    return board, side_before, moves_prefix

In [ ]:
def clip_cp(cp: float, clip_value=RAW_CLIP_CP):
    return float(max(-clip_value, min(clip_value, cp)))


def normalize_cp_smooth(cp: float, scale=SMOOTH_SCALE, target_range=TARGET_RANGE):
    return np.float32(target_range * (cp / (scale + abs(cp))))


def normalize_cp_tanh(cp: float, scale=TANH_SCALE, target_range=TARGET_RANGE):
    return np.float32(target_range * np.tanh(cp / scale))


def apply_normalization(cp: float, mode=NORMALIZATION_MODE):
    if mode == "none":
        return np.float32(cp)
    elif mode == "clip":
        return np.float32(clip_cp(cp))
    elif mode == "smooth":
        return normalize_cp_smooth(cp)
    elif mode == "tanh":
        return normalize_cp_tanh(cp)
    else:
        raise ValueError(f"Unknown NORMALIZATION_MODE: {mode}")


def stm_score_to_red_score(stm_score: int, side_to_move: str) -> int:
    return stm_score if side_to_move == "red" else -stm_score

In [ ]:
class PikafishEngine:
    def __init__(self, path, threads=1, hash_mb=128):
        self.path = path
        self.threads = threads
        self.hash_mb = hash_mb

        self.p = subprocess.Popen(
            [path],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        self.send("uci")
        self._wait_for("uciok")

        self.send(f"setoption name Threads value {threads}")
        self.send(f"setoption name Hash value {hash_mb}")

        self.send("isready")
        self._wait_for("readyok")

    def send(self, cmd: str):
        self.p.stdin.write(cmd + "\n")
        self.p.stdin.flush()

    def _wait_for(self, token: str):
        while True:
            line = self.p.stdout.readline()
            if not line:
                continue
            line = line.strip()
            if token in line:
                return line

    def new_game(self):
        self.send("ucinewgame")
        self.send("isready")
        self._wait_for("readyok")

    def _position_cmd(self, moves_prefix):
        if moves_prefix:
            return "position startpos moves " + " ".join(moves_prefix)
        return "position startpos"

    def evaluate_moves_prefix_stm(self, moves_prefix, depth=10, max_nodes=50000):
        """
        Returns score from side-to-move perspective at the given position.
        Uses one engine only, no multipv.
        Stops on bestmove.
        """
        self.new_game()
        self.send(self._position_cmd(moves_prefix))
        self.send(f"go depth {depth} nodes {max_nodes}")

        last_score_cp = None
        last_score_mate = None
        bestmove = None

        while True:
            line = self.p.stdout.readline()
            if not line:
                continue
            line = line.strip()

            if line.startswith("info"):
                m_cp = re.search(r"\bscore cp (-?\d+)", line)
                m_mate = re.search(r"\bscore mate (-?\d+)", line)

                if m_cp:
                    last_score_cp = int(m_cp.group(1))
                    last_score_mate = None
                elif m_mate:
                    last_score_mate = int(m_mate.group(1))
                    last_score_cp = None

            elif line.startswith("bestmove"):
                parts = line.split()
                if len(parts) >= 2:
                    bestmove = parts[1]
                break

        if last_score_mate is not None:
            return 30000 if last_score_mate > 0 else -30000, bestmove

        if last_score_cp is None:
            return 0, bestmove

        return int(last_score_cp), bestmove

    def close(self):
        try:
            self.send("quit")
        except:
            pass
        try:
            self.p.terminate()
        except:
            pass

In [ ]:
source_games_by_index = {idx: g for idx, g in enumerate(source_games)}

missing_game_indices = sorted(set(df["game_index"].unique()) - set(source_games_by_index.keys()))
print("missing game indices:", len(missing_game_indices))

if len(missing_game_indices) > 0:
    print("first missing indices:", missing_game_indices[:20])

In [ ]:
def label_sampled_moves(df, source_games_by_index, pikafish_path, depth, max_nodes, threads=1, hash_mb=256):
    engine = PikafishEngine(
        path=pikafish_path,
        threads=threads,
        hash_mb=hash_mb
    )

    rows = []
    tensors = []

    try:
        total = len(df)

        for i, row in enumerate(df.itertuples(index=False), start=1):
            game_index = int(row.game_index)
            ply_index = int(row.ply_index_0based)
            sampled_move = str(row.move).strip().lower()

            if game_index not in source_games_by_index:
                raise KeyError(f"game_index {game_index} not found in source_games_by_index")

            game = source_games_by_index[game_index]
            game_moves = game["moves"]

            if ply_index < 0 or ply_index >= len(game_moves):
                raise IndexError(
                    f"row {i}: ply_index {ply_index} out of range for game_index {game_index} "
                    f"(num_plies={len(game_moves)})"
                )

            true_move = game_moves[ply_index]
            if sampled_move != true_move:
                raise ValueError(
                    f"row {i}: sampled move mismatch for game_index {game_index}, "
                    f"ply_index {ply_index}. csv={sampled_move}, source={true_move}"
                )
            
            board_before, side_before, moves_prefix_before = reconstruct_position_before_ply(
                game_moves=game_moves,
                ply_index_0based=ply_index
            )

            board_after = clone_board(board_before)
            apply_move(board_after, sampled_move)
            side_after = opposite_side(side_before)
            moves_prefix_after = moves_prefix_before + [sampled_move]

            stm_score_after, engine_bestmove = engine.evaluate_moves_prefix_stm(
                moves_prefix=moves_prefix_after,
                depth=depth,
                max_nodes=max_nodes
            )

            raw_red_cp = stm_score_to_red_score(stm_score_after, side_after)
            raw_red_cp_clipped = clip_cp(raw_red_cp)
            norm_target = apply_normalization(raw_red_cp_clipped)

            tensor = board_to_tensor(board_after, side_after)
            tensors.append(tensor)

            rows.append({
                "game_index": game_index,
                "event": row.event,
                "red": row.red,
                "black": row.black,
                "stage": row.stage,
                "ply_index_0based": ply_index,
                "ply_number_1based": int(row.ply_number_1based),
                "move": sampled_move,

                "side_before_move": side_before,
                "side_after_move": side_after,

                "engine_bestmove_after_position": engine_bestmove,
                "raw_stm_score_after_move": stm_score_after,
                "raw_red_cp_after_move": raw_red_cp,
                "raw_red_cp_clipped": raw_red_cp_clipped,
                "target_normalized": float(norm_target),
            })

            if i % 100 == 0 or i == total:
                print(f"processed {i}/{total}")

    finally:
        engine.close()

    out_df = pd.DataFrame(rows)
    X = np.array(tensors, dtype=np.float32)

    return out_df, X

In [ ]:
labeled_df, X = label_sampled_moves(
    df=df,
    source_games_by_index=source_games_by_index,
    pikafish_path=PIKAFISH_PATH,
    depth=LABEL_DEPTH,
    max_nodes=LABEL_MAX_NODES,
    threads=ENGINE_THREADS,
    hash_mb=ENGINE_HASH_MB
)

print("labeled rows:", len(labeled_df))
print("tensor shape:", X.shape)
display(labeled_df.head())

In [ ]:
raw_df = labeled_df[[
    "game_index",
    "event",
    "red",
    "black",
    "stage",
    "ply_index_0based",
    "ply_number_1based",
    "move",
    "side_before_move",
    "side_after_move",
    "engine_bestmove_after_position",
    "raw_stm_score_after_move",
    "raw_red_cp_after_move",
    "raw_red_cp_clipped",
]].copy()

norm_df = labeled_df[[
    "game_index",
    "event",
    "red",
    "black",
    "stage",
    "ply_index_0based",
    "ply_number_1based",
    "move",
    "side_before_move",
    "side_after_move",
    "engine_bestmove_after_position",
    "raw_red_cp_clipped",
    "target_normalized",
]].copy()

raw_df.to_csv(OUTPUT_RAW_CSV, index=False)
norm_df.to_csv(OUTPUT_NORM_CSV, index=False)

np.savez_compressed(
    OUTPUT_RAW_NPZ,
    X=X,
    y_raw=labeled_df["raw_red_cp_clipped"].to_numpy(dtype=np.float32)
)

np.savez_compressed(
    OUTPUT_NORM_NPZ,
    X=X,
    y=labeled_df["target_normalized"].to_numpy(dtype=np.float32)
)

print("saved raw csv :", OUTPUT_RAW_CSV)
print("saved norm csv:", OUTPUT_NORM_CSV)
print("saved raw npz :", OUTPUT_RAW_NPZ)
print("saved norm npz:", OUTPUT_NORM_NPZ)

In [ ]:
print("raw clipped min:", float(labeled_df["raw_red_cp_clipped"].min()))
print("raw clipped max:", float(labeled_df["raw_red_cp_clipped"].max()))
print("norm min       :", float(labeled_df["target_normalized"].min()))
print("norm max       :", float(labeled_df["target_normalized"].max()))

print("\nstage counts:")
print(labeled_df["stage"].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(labeled_df["raw_red_cp_clipped"], bins=NUM_HIST_BINS)
plt.title("Raw Red-Perspective Score Distribution")
plt.xlabel("Raw clipped cp")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()

for stage_name in ["early", "mid", "end"]:
    stage_df = labeled_df[labeled_df["stage"] == stage_name]

    plt.figure(figsize=(8, 5))
    plt.hist(stage_df["raw_red_cp_clipped"], bins=NUM_HIST_BINS)
    plt.title(f"Raw Score Distribution - {stage_name}")
    plt.xlabel("Raw clipped cp")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.show()

stage_summary = labeled_df.groupby("stage").agg(
    count=("move", "count"),
    raw_min=("raw_red_cp_clipped", "min"),
    raw_max=("raw_red_cp_clipped", "max"),
    raw_mean=("raw_red_cp_clipped", "mean"),
    raw_std=("raw_red_cp_clipped", "std"),
    norm_min=("target_normalized", "min"),
    norm_max=("target_normalized", "max"),
    norm_mean=("target_normalized", "mean"),
    norm_std=("target_normalized", "std"),
).reset_index()

display(stage_summary)